# 서울 9호선 Lakehouse 탐색 (PySpark)

Spark 로 **Iceberg Silver/Gold**(지연 분석)와 **Paimon Bronze**(원본/현재상태)를 직접 조회·변환한다.
StarRocks/Grafana 가 미리 만든 집계 말고, 분석가가 SQL/DataFrame 으로 자유롭게 파보는 환경.

> 첫 셀 실행 시 커넥터 jar 를 ivy 로 받는다(이미 받았으면 캐시에서 즉시).

In [3]:
from pyspark.sql import SparkSession, functions as F

PKGS = ",".join([
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
    "org.apache.iceberg:iceberg-aws-bundle:1.6.1",
    "org.apache.paimon:paimon-spark-3.5_2.12:1.4.1",
    "org.apache.paimon:paimon-s3:1.4.1",
])
NETTY = "-Dorg.apache.iceberg.shaded.io.netty.noUnsafe=true -Dio.netty.noUnsafe=true"

spark = (
    SparkSession.builder.appName("jupyter-explore")
    .config("spark.jars.packages", PKGS)
    .config("spark.driver.extraJavaOptions", NETTY)
    .config("spark.sql.iceberg.vectorization.enabled", "false")
    # Iceberg REST 카탈로그 (Silver/Gold)
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "rest")
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true")
    .config("spark.sql.catalog.iceberg.warehouse", "s3://warehouse/")
    # Paimon 카탈로그 (Bronze)
    .config("spark.sql.catalog.paimon", "org.apache.paimon.spark.SparkCatalog")
    .config("spark.sql.catalog.paimon.warehouse", "s3://paimon/warehouse")
    .config("spark.sql.catalog.paimon.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.paimon.s3.access-key", "minioadmin")
    .config("spark.sql.catalog.paimon.s3.secret-key", "minioadmin")
    .config("spark.sql.catalog.paimon.s3.path.style.access", "true")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
        "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions",
    )
    .getOrCreate()
)
print(spark.version)

3.5.6


In [4]:
# 어떤 네임스페이스/테이블이 있나
spark.sql("SHOW NAMESPACES IN iceberg").show()
spark.sql("SHOW TABLES IN iceberg.gold").show()

+---------+
|namespace|
+---------+
|     demo|
|      dim|
|     gold|
|   silver|
+---------+

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     gold| delay_by_station|      false|
|     gold|delay_by_timeband|      false|
+---------+-----------------+-----------+



## Gold 조회 — 연착 빈번 역 Top5

In [5]:
spark.sql("""
    SELECT station_nm, arrivals, delayed, delay_rate, avg_delay_sec
    FROM iceberg.gold.delay_by_station
    ORDER BY delayed DESC, delay_rate DESC
    LIMIT 5
""").toPandas()

,station_nm,arrivals,delayed,delay_rate,avg_delay_sec
0,종합운동장,3,2,0.6667,37.3
1,송파나루,1,1,1.0000,286.0
2,구반포,1,1,1.0000,286.0
3,언주,2,1,0.5000,-2.0
4,삼전,2,1,0.5000,10.0


In [6]:
# 시간대 × 요일유형 지연율
spark.sql("""
    SELECT day_type, time_band, arrivals, delayed, delay_rate, avg_delay_sec
    FROM iceberg.gold.delay_by_timeband
    ORDER BY day_type, time_band
""").toPandas()

,day_type,time_band,arrivals,delayed,delay_rate,avg_delay_sec
0,평일,점심,87,19,0.2184,634.5


## 직접 변환 예시 — Silver 에서 방향별 지연 통계

미리 만든 Gold 가 아니라, Silver(`arrival_delay`)를 DataFrame 으로 자유롭게 가공해본다.

In [7]:
silver = spark.table("iceberg.silver.arrival_delay")
(
    silver.groupBy("direction")
    .agg(
        F.count("*").alias("arrivals"),
        F.sum("is_delayed").alias("delayed"),
        F.round(F.avg("delay_sec"), 1).alias("avg_delay_sec"),
    )
    .toPandas()
)

,direction,arrivals,delayed,avg_delay_sec
0,하행,46,12,1243.1
1,상행,41,7,-48.3


In [8]:
# Bronze(Paimon) 도 같은 세션에서 조회 가능 — 원본 로그 행수
spark.sql("SELECT COUNT(*) AS log_rows FROM paimon.bronze.subway_position_log").show()

26/06/16 08:17:49 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods


+--------+
|log_rows|
+--------+
|    1780|
+--------+

